# Zazo — تدريب Gemma 2B LoRA فقط

هذا الدفتر لا يحتوي على Cloudflare أو KV أو Android. وظيفته فقط: قراءة ملف التدريب، تدريب LoRA، ثم تجهيز الأوزان للتنزيل أو الحفظ في Google Drive.

الصيغة المفضلة لملف التدريب هي JSONL، وكل سطر يحتوي `instruction` و`response`. يلزم GPU وHF_TOKEN لأن Gemma نموذج مقيد الوصول. التوكن يُستخدم في الذاكرة للجلسة الحالية فقط ولا يُحفظ.

In [ ]:
%pip install -q -U transformers datasets peft accelerate bitsandbytes huggingface_hub safetensors
print('تم تثبيت الحزم')

In [ ]:
import os, json, shutil, zipfile, getpass
from pathlib import Path
import torch

print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('اختر GPU من إعدادات البيئة قبل المتابعة.')
print('GPU:', torch.cuda.get_device_name(0))

MODEL_NAME = 'google/gemma-2b-it'
OUTPUT_DIR = Path('/kaggle/working/zazo_lora') if Path('/kaggle/working').exists() else Path('/content/zazo_lora')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_LENGTH = 512
EPOCHS = 1
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8
print('مجلد الناتج:', OUTPUT_DIR)

In [ ]:
from pathlib import Path
SUPPORTED = {'.jsonl', '.ndjson', '.json', '.csv', '.tsv'}

def locate_training_file():
    roots = [Path('/kaggle/input'), Path('/content')]
    candidates = [p for root in roots if root.exists() for p in root.rglob('*') if p.is_file() and p.suffix.lower() in SUPPORTED]
    if candidates:
        return max(candidates, key=lambda p: p.stat().st_mtime)
    try:
        from google.colab import files
        print('اختر ملف التدريب JSONL أو JSON أو CSV أو TSV')
        uploaded = files.upload()
        for name, content in uploaded.items():
            path = Path('/content') / Path(name).name
            if path.suffix.lower() in SUPPORTED:
                path.write_bytes(content)
                return path
    except ImportError:
        pass
    raise RuntimeError('لم يتم العثور على ملف تدريب. أضفه إلى Kaggle كـDataset أو شغّل الدفتر في Colab واختر الملف.')

TRAIN_FILE = locate_training_file()
print('ملف التدريب:', TRAIN_FILE)

In [ ]:
import csv, json
from datasets import Dataset

def load_records(path):
    text = path.read_text(encoding='utf-8-sig')
    if path.suffix.lower() in {'.jsonl', '.ndjson'}:
        rows = [json.loads(line) for line in text.splitlines() if line.strip()]
    elif path.suffix.lower() == '.json':
        value = json.loads(text)
        rows = value if isinstance(value, list) else value.get('records', value.get('data', []))
    else:
        delimiter = '\t' if path.suffix.lower() == '.tsv' else ','
        rows = list(csv.DictReader(text.splitlines(), delimiter=delimiter))
    if not rows or not isinstance(rows[0], dict):
        raise RuntimeError('ملف التدريب فارغ أو ليس قائمة سجلات.')
    formatted = []
    for row in rows:
        instruction = row.get('instruction', row.get('question', row.get('prompt', '')))
        response = row.get('response', row.get('answer', row.get('output', '')))
        if instruction and response:
            formatted.append({'text': f'Instruction: {instruction}\nResponse: {response}'})
    if not formatted:
        raise RuntimeError('كل سجل يحتاج instruction وresponse، أو question وanswer، أو prompt وoutput.')
    return Dataset.from_list(formatted)

dataset = load_records(TRAIN_FILE)
print('عدد أمثلة التدريب:', len(dataset))
print(dataset[0])

In [ ]:
# ضع توكن Hugging Face هنا للجلسة الحالية فقط. لا تضعه في GitHub أو تشارك الدفتر بعد تعبئة القيمة.
HF_TOKEN = ""  # مثال: hf_xxxxxxxxxxxxxxxxx

from huggingface_hub import login
if not HF_TOKEN.strip():
    import getpass
    HF_TOKEN = getpass.getpass('HF_TOKEN من Hugging Face (يبدأ بـ hf_): ').strip()
if not HF_TOKEN.startswith('hf_'):
    raise RuntimeError('ضع توكن Hugging Face صحيحًا يبدأ بـ hf_.')
login(token=HF_TOKEN, add_to_git_credential=False)
print('تم تسجيل الدخول للجلسة الحالية فقط؛ لن يتم حفظ التوكن في Google Drive.')


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, token=HF_TOKEN, torch_dtype=torch.float16, device_map='auto'
)
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    task_type=TaskType.CAUSAL_LM, bias='none'
))
model.print_trainable_parameters()

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)
tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR), num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=2e-4, logging_steps=1, save_strategy='no',
    fp16=True, report_to='none', optim='paged_adamw_8bit',
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized,
                  data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))
trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('اكتمل التدريب وحُفظت الأوزان في:', OUTPUT_DIR)

In [ ]:
ZIP_PATH = OUTPUT_DIR.parent / 'zazo_lora_weights.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as archive:
    for file in OUTPUT_DIR.rglob('*'):
        if file.is_file():
            archive.write(file, file.relative_to(OUTPUT_DIR.parent))
print('ملف الأوزان المضغوط:', ZIP_PATH)

try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except ImportError:
    print('في Kaggle: افتح تبويب Output ثم نزّل zazo_lora_weights.zip')

# اختياري في Colab: حفظ نسخة في Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_path = Path('/content/drive/MyDrive/zazo_lora_weights.zip')
    shutil.copy2(ZIP_PATH, drive_path)
    print('تم النسخ إلى Google Drive:', drive_path)
except Exception as e:
    print('تخطي Google Drive:', e)